# GS-DroneGym Colab PointNav GPU Debug

This notebook is for debugging the failing **Live PointNav** loop on Google Colab.

It separates three questions:

1. Does the environment run?
2. Can simple policies be evaluated live?
3. Does a trained BC policy actually improve closed-loop navigation?

The notebook starts with the CPU/mock renderer because that is the fastest reliable debug path. GPU/real `gsplat` scenes are optional cells later.


## 0. Colab Runtime Setup

In Colab, select:

`Runtime -> Change runtime type -> T4 GPU` or better.

A GPU is useful for visual training and real `gsplat` rendering, but PointNav control debugging should first pass on the mock path.


In [ ]:
!nvidia-smi || true
import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())


## 1. Install GS-DroneGym

Use PyPI for the published package. Use the GitHub install if you need the newest repo changes before the next PyPI release.


In [ ]:
USE_GITHUB_MAIN = True

if USE_GITHUB_MAIN:
    !pip install -q --upgrade "git+https://github.com/09Catho/gs-dronegym.git"
else:
    !pip install -q --upgrade gs-dronegym


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import torch

import gs_dronegym
from gs_dronegym.benchmarks import make_benchmark

print('gs_dronegym version:', gs_dronegym.__version__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 2. Smoke Test PointNav

This checks that `PointNav-v0` creates observations and can step in the live Gym loop.


In [ ]:
env = gs_dronegym.make('PointNav-v0', scene=None, image_size=(96, 96))
obs, info = env.reset(seed=0)
print('obs keys:', obs.keys())
print('rgb:', obs['rgb'].shape, obs['rgb'].dtype)
print('depth:', obs['depth'].shape, obs['depth'].dtype)
print('state:', obs['state'].shape, obs['state'].dtype)
print('instruction:', obs['instruction'])
print('goal:', info.get('goal_position'), 'distance:', info.get('distance_to_goal'))

obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print('step:', type(reward), terminated, truncated, info)
env.close()


## 3A. Expert Control Check

This is the key Live PointNav sanity check. If the expert succeeds, the environment/controller/action interface is solvable. If a learned BC policy fails later, the problem is training/data, not the simulator loop.


In [ ]:
!gs-dronegym-evaluate-expert \
  --env-id PointNav-v0 \
  --scene None \
  --n-episodes 10 \
  --width 64 --height 64 \
  --renderer-device cpu \
  --output /content/pointnav_expert_eval.json

print(Path('/content/pointnav_expert_eval.json').read_text())


## 3. Evaluate Zero and Random Policies

These are negative controls. They should usually fail. The point is to verify that live evaluation works and to inspect failure modes.


In [ ]:
def compact_report(report):
    d = report.to_dict()
    return {
        'success_rate': d['core_metrics']['success_rate'],
        'mean_return': d['core_metrics']['mean_return'],
        'mean_episode_length': d['core_metrics']['mean_episode_length'],
        'collision_rate': d['benchmark_metrics']['collision_rate'],
        'spl': d['benchmark_metrics']['spl'],
        'mean_path_length': d['benchmark_metrics']['mean_path_length'],
    }

benchmark = make_benchmark('gs_dronegym', env_id='PointNav-v0', scene=None, image_size=(96, 96))
zero_report = benchmark.evaluate_policy(policy=None, n_episodes=3, seed=10)
print('zero:', json.dumps(compact_report(zero_report), indent=2))

rng = np.random.default_rng(123)
def random_policy(_obs):
    return rng.uniform(-1, 1, size=4).astype(np.float32)

random_report = benchmark.evaluate_policy(policy=random_policy, n_episodes=3, seed=20)
print('random:', json.dumps(compact_report(random_report), indent=2))


## 4. Visualize One Live Rollout

This plots the top-down path and distance-to-goal for a random policy. If BC fails later, use the same plots to see whether it dives, saturates actions, or flies away from the target.


In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
env = gs_dronegym.make('PointNav-v0', scene=None, image_size=(96, 96))
obs, info = env.reset(seed=1)
positions = [obs['state'][:3].copy()]
distances = [float(info.get('distance_to_goal', np.nan))]
actions = []
frames = [obs['rgb'].copy()]

for step in range(80):
    action = rng.uniform(-1, 1, size=4).astype(np.float32)
    obs, reward, terminated, truncated, info = env.step(action)
    positions.append(obs['state'][:3].copy())
    distances.append(float(info.get('distance_to_goal', np.nan)))
    actions.append(action)
    frames.append(obs['rgb'].copy())
    if terminated or truncated:
        break

env.close()
positions = np.asarray(positions)
actions = np.asarray(actions)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(frames[0])
axes[0].set_title('first RGB frame')
axes[0].axis('off')
axes[1].plot(positions[:, 0], positions[:, 1], marker='o', markersize=2)
axes[1].scatter(positions[0, 0], positions[0, 1], c='green', label='start')
axes[1].scatter(positions[-1, 0], positions[-1, 1], c='red', label='end')
axes[1].set_title('top-down XY path')
axes[1].axis('equal')
axes[1].legend()
axes[2].plot(distances)
axes[2].set_title('distance to goal')
axes[2].set_xlabel('step')
axes[2].set_ylabel('meters')
plt.tight_layout()
plt.show()

print('steps:', len(positions) - 1)
print('final info:', info)
print('action min/max:', actions.min(axis=0) if len(actions) else None, actions.max(axis=0) if len(actions) else None)


## 5. Generate A PointNav-Focused Dataset

The current package generator samples a curriculum mix. For now, this creates a small mock dataset to prove the train/eval pipeline. For serious PointNav success, we should add a PointNav-only generator mode in the repo.


In [ ]:
DATASET_ROOT = Path('/content/pointnav_debug_dataset')
!rm -rf {DATASET_ROOT}

!gs-dronegym-generate-dataset {DATASET_ROOT} \
  --scenes mock://colab_a mock://colab_b mock://colab_c \
  --episodes-per-scene 8 \
  --width 96 --height 96 \
  --renderer-device cpu \
  --allow-mock-rendering \
  --dataset-id colab_pointnav_debug \
  --task-filter point_nav

!gs-dronegym-validate-dataset {DATASET_ROOT}


## 6. Train Behavior Cloning On GPU If Available

This trains the included BC baseline. The key metric is not just train loss. We also evaluate live after training.


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT = Path('/content/pointnav_debug_bc.pt')

!gs-dronegym-train-bc {DATASET_ROOT} \
  --format gs_dronegym \
  --epochs 5 \
  --batch-size 16 \
  --device {DEVICE} \
  --checkpoint {CHECKPOINT}


## 7. Evaluate The BC Policy Live

If this fails, that means the learned policy is not yet good enough for closed-loop navigation. That is expected for tiny mock data. The next fix is PointNav-only data + recovery perturbations + state-only baseline.


In [ ]:
!gs-dronegym-evaluate \
  --benchmark gs_dronegym \
  --env-id PointNav-v0 \
  --scene None \
  --policy {CHECKPOINT} \
  --n-episodes 5 \
  --device {DEVICE} \
  --output /content/pointnav_bc_eval.json

print(Path('/content/pointnav_bc_eval.json').read_text()[:2000])


## 8. Optional: Real 3DGS Scene Smoke Test

This may download multiple GB. Do not run unless you have enough Colab disk and time.

The built-in handles are:

- `room`
- `garden`
- `bicycle`
- `truck`

For first real-scene testing, use `room` because it is smaller than the outdoor scenes.


In [ ]:
RUN_REAL_SCENE_DOWNLOAD = False

if RUN_REAL_SCENE_DOWNLOAD:
    from gs_dronegym.scene import get_scene
    scene_path = get_scene('room')
    print('cached scene:', scene_path)

    env = gs_dronegym.make('PointNav-v0', scene='room', renderer_device='cuda', image_size=(224, 224))
    obs, info = env.reset(seed=0)
    print(obs['rgb'].shape, obs['depth'].shape, info)
    env.close()
else:
    print('Skipping real scene download. Set RUN_REAL_SCENE_DOWNLOAD=True to test gsplat scenes.')


## 9. What To Do If Live PointNav Still Fails

If BC fails live, do not immediately blame GPU. Check these in order:

1. expert/controller success on PointNav
2. state-only BC success
3. action saturation in `[-1, 1]`
4. PointNav-only dataset size
5. recovery perturbation data
6. live evaluation every epoch
7. only then real 3DGS rendering

A GPU helps speed, but it does not fix closed-loop policy errors by itself.
